# Diabetes Challenge - Solution

This notebook works through the diabetes challenge following the [machine learning workflow](../machine_learning_workflow.md): define the goal and the success metric, get and clean the data, explore it, engineer features, then train, tune, and evaluate models.


> [!NOTE]
> This solution handles the class imbalance with a **before-training** strategy: it synthesizes new minority-class samples with `SMOTE` before training (see Step 5). As an alternative, you can use a **during-training** strategy that lets the algorithm account for the imbalance itself, for example setting `class_weight='balanced'` on a `DecisionTreeClassifier` (or on a `LogisticRegression` if you use one). Either approach is valid; this notebook shows the sampling route.

## Step 1 - Define the Goal and Research Question

The goal is to predict whether a patient has diabetes from a set of medical measurements. The target is `Outcome`, where 1 means the patient tested positive for diabetes.

### Success Metric

We decide how to measure success before modelling. This is a medical screening task, so a **false negative** (telling a patient who has diabetes that they are healthy) is more costly than a **false positive** (a false alarm we can follow up with another test). Plain accuracy treats both errors the same, so we focus on **recall** for the positive class (Recall = TP / (TP + FN)), the share of true diabetes cases we catch. To avoid ignoring precision completely, we select and tune on the **F2 score**: an F-beta score with beta = 2, which weights recall higher than precision.

### Data Overview

The dataset has several medical predictor variables and one target variable, `Outcome`. The predictors include the number of pregnancies, BMI, insulin level, age, and so on. For `Outcome`, a value of 1 means the patient tested positive for diabetes.

|Column Name| Description|
|:------------|:------------|
|Pregnancies|Number of times pregnant|
|Glucose|Plasma glucose concentration a 2 hours in an oral glucose tolerance test|
|BloodPressure|Diastolic blood pressure (mm Hg)|
|SkinThickness|Triceps skin fold thickness (mm)|
|Insulin|2-Hour serum insulin (mu U/ml)|
|BMI|Body mass index (weight in kg/(height in m)^2)|
|DiabetesPedigreeFunction| Diabetes pedigree function|
|Age| Age (years)|
|Outcome|Class variable (0 or 1) |

## Step 2 - Get the Data

We import the libraries we need, then pull the diabetes data from the database.

### Set-up and Import

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Libraries for data import
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Preprocessing
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import recall_score, accuracy_score, precision_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
from sklearn.metrics import fbeta_score, make_scorer


# Define a plotting style to be used for all plots in this notebook
plt.style.use('tableau-colorblind10')

### Load from the Database

The data lives in our Postgres database in the `diabetes` schema, split across several tables (`patient`, `blood_metrics`, `skin`, `pedigree_outcome`). We join them on the patient id to assemble one row per patient. After loading we can optionally cache the result to a CSV so we do not have to query the database on every run.

In [ ]:
# Read the database connection string from the .env file
load_dotenv()

DB_STRING = os.getenv('DB_STRING')
db = create_engine(DB_STRING)

In [ ]:
# Pull the data straight from the database, joining the tables on the patient id
query_string = """
SET schema 'diabetes';

SELECT *
FROM patient p
LEFT JOIN blood_metrics bd ON p.id = bd.patientid
LEFT JOIN skin s ON p.id = s.patientid
LEFT JOIN pedigree_outcome po ON p.id = po.PatientId
WHERE bd.measurement_date = '2022-12-13';
"""

df = pd.read_sql(query_string, db)

In [ ]:
# A SELECT * across the joins brings in the join keys (id, patientid) and the
# measurement_date column, and Postgres lower-cases the names. Drop the keys and
# restore the feature names the rest of the notebook expects.
df = df.drop(columns=['id', 'patientid', 'measurement_date'])
df = df.rename(columns={
    'pregnancies': 'Pregnancies',
    'glucose': 'Glucose',
    'bloodpressure': 'BloodPressure',
    'skinthickness': 'SkinThickness',
    'insulin': 'Insulin',
    'bmi': 'BMI',
    'diabetespedigreefunction': 'DiabetesPedigreeFunction',
    'outcome': 'Outcome',
})

In [ ]:
# Optional: cache the data locally so you do not have to query the database every run
# df.to_csv('data/diabetes.csv', index=False)

In [ ]:
df.head()

### First Look at the Data

Before building a model we take a quick look at the data to understand its size and spot anything that needs cleaning.

In [ ]:
#Print the shape of the data
print('Diabetes dataset')
print('==================')
print('# observations: {}'.format(df.shape[0]))
print('# features:     {}'.format(df.shape[1]-1))

As we can see, the dataset contains 768 observations and 9 columns from which the last column defines the label, i.e. the test result. 
Thus, we have 8 independent features.

Let us take a look at the single variables included in the dataset:

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

From the output above we could assume that the data is already rather clean with no missing values, but the summary we get with `.describe()` should make us wonder... We definitely have to check if in our case maybe missing values are represented by zeros. Before we have a closer look at the data and try to impute values, we will split our data into train and test set to make sure our test data will be not "contaminated" by information from the train data and vice versa.

## Step 3 - Train-Test Split

[Stratification](https://en.wikipedia.org/wiki/Stratified_sampling) is a statistical method that allows us to sample observations from data that is partitioned into subgroups. Here: subgroups defined by the label, i.e. a positive or negative test result. Using this method, we can ensure that the data distributions in the training and test sets are nearly the same. This is useful because the data distribution we want to apply the model to later should be identical to the one we train the model on. 

For the test set we will use 33% of the available data. In cases where the amount of data is limited we should carefully separate enough data into the test set so we can ensure later that the model generalizes well enough to new data. The ratio of between training and test data increases with the amount of data. In case of a large data set - say 10 million entries - we can easily reduce the test set to only 1% of the data - still 100,000 observations.

In the train-test split we apply stratification which samples us the train and test data in such a way that the same ratios of positive and negative labels occur in train and test data. To get the ratio to be applied in stratification we need to provide the data along which the ratio that should be computed - here the labels. 

In [ ]:
# Stratified train-test-split
df_train, df_test= train_test_split(df, test_size=0.33, random_state=42, stratify=df.Outcome)

In [ ]:
# Print shape of datasets
print('Train data')
print('# df_train:     {}'.format(df_train.shape[0]))
print('==================')
print('Test data')
print('# df_test:     {}'.format(df_test.shape[0]))

In [ ]:
# Visualize y_train and y_test
fig, axes = plt.subplots(1,2, figsize=(10, 4))
fig.suptitle("Representation of target variable: Train (left) vs. Test (right) data set")
sns.countplot(x=df_train.Outcome, ax=axes[0]);
sns.countplot(x=df_test.Outcome, ax=axes[1]);

## Step 4 - Data Exploration

We explore the training data to understand the distributions, find the hidden missing values (zeros that should not be there), and check how the features relate to each other and to the target. The cleaning decisions we make here are carried over to the test set in the next step.

In [ ]:
# Plot distribution of features 
features = df_train.columns.tolist()
features.remove('Outcome')

fig,ax = plt.subplots(3,3,figsize=(16,12))
count = 0
for item in features:
    sns.histplot(df_train[item], kde=True, ax=ax[int(count/3)][count%3], color='#33658A').set(title=item, xlabel='')
    count += 1
ax.flat[-1].set_visible(False)
fig.tight_layout(pad=3)

These plots clearly show several things:
* The distributions of `Pregnancies`, `Age`, `Insulin` and `DiabetesPedigreeFunction` are positively/right skewed. Transforming them might increase our model's performance (depending on the model type we use).
* The features `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` include zeros, even if it is not a expected value for those features. This might indicate, that missing values are represented as zeros in this dataset. We have to find a suitable strategy to replace/impute those values. 

### Analysis of Missing Values (== 0)

In [ ]:
# Replace zeros with np.nan
def replace_zeros(df, feature):
    df[feature] = df[feature].replace(0, np.nan)
    return df

columns_with_zeros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in columns_with_zeros:
    df_train = replace_zeros(df_train, col)

In [ ]:
# Check for missing data
df_train.isnull().sum()

Insulin and SkinThickness have a lot of missing values. Let's start by looking at the columns with fewer missings and try to replace/drop them.

In [ ]:
print(df_train.query("BMI.isnull() & BloodPressure.isnull()").shape[0])
index_list = df_train.query("BMI.isnull() & BloodPressure.isnull()").index.tolist()
df_train.query("BMI.isnull() & BloodPressure.isnull()")

There are 5 observations with missing values in 4 of 8 features. We will drop those rows.

In [ ]:
# Drop rows by index
df_train = df_train.drop(index_list)

In [ ]:
# Plot distribution of numerical features with missing values 
plot_col = ['BMI', 'BloodPressure', 'Glucose']
fig,ax = plt.subplots(1, 3,figsize=(16,4))
count = 0
for col in plot_col:
    sns.histplot(df_train[col], kde=True, ax=ax[count], color='#33658A').set(title=col, xlabel='')
    #plt.axvlines(X_train[col].mean(), 0, X_train[col].max(), color="red")
    count += 1
fig.tight_layout(pad=3)

In [ ]:
# Print mean and median for those features
for col in plot_col:
    print(f"{col} mean: {df_train[col].mean().round(2)}")
    print(f"{col} median: {df_train[col].median().round(2)}")

For a first try, we will replace the missing values for those columns either with the mean or median. It makes sense to check the distribution to decide which value to choose. Comparing the mean and median for those columns shows, that both values for BMI and BloodPressure are pretty close together. Only the heavies skewed column Glucose shows a remarkable difference between mean and median. We will keep it simple and replace all missing values with the median of the respective features. 

In [ ]:
# Replace missing values with median of respective feature
values_dict = {}
for col in plot_col:
    values_dict[col] = df_train[col].median()
    df_train[col] = df_train[col].fillna(values_dict[col])

In [ ]:
# Check for missings again
df_train.isnull().sum()

In [ ]:
print("SkinThickness % of missing data:", ((df_train.SkinThickness.isnull().sum()/df_train.shape[0])*100).round(2))
print("Insulin % of missing data:", ((df_train.Insulin.isnull().sum()/df_train.shape[0])*100).round(2))

There are still two columns where a high proportion of the data is missing. Replacing them with the mean or median might induce inaccuracies. We can have a look if there are other features, which are strongly correlated so that we can drop the ones with the missing values.

### Correlations

In [ ]:
sns.pairplot(df_train[["Glucose", "SkinThickness", "BMI", "Insulin", "Outcome"]], hue="Outcome");

In [ ]:
# Calculate correlations between features
corr = df_train.corr()

plt.subplots(figsize=(15, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap="YlGnBu_r", mask=mask, vmax=1, vmin=-1);

From the pairplot and correlation heatmap above we can see that the features `BMI` and `SkinThickness` are rather highly correlated. The same applies for the features `Glucose` and `Insulin`. Since we have so many missing values for `SkinThickness` and `Insulin` we will drop them now.

In [ ]:
# Drop Insulin and SkinThickness column
df_train = df_train.drop(["Insulin", "SkinThickness"], axis=1)

In [ ]:
df_train.head(2)

In [ ]:
# Check for missings AGAIN
df_train.isnull().sum().sum()

## Step 5 - Feature Engineering

With the exploration done, we prepare the features. First we apply the same missing-value handling to the test set using values learned on the train set. Scaling and class balancing are then set up to run inside the modelling pipeline, so they are refit on the training part of each cross-validation fold and never leak information from the validation fold.

### Preparing the Test Set

We have replaced missing values and dropped some columns in our train data. We need to prepare our test data the same way to be able to make proper predictions. To fill in the missing values we will use the median we calculated on the training set and stored in `values_dict`.

In [ ]:
df_test.head(2)

In [ ]:
# Convert zeros to nan values
for col in columns_with_zeros:
    df_test = replace_zeros(df_test, col)

df_test.isnull().sum()

In [ ]:
# Replace missing values in test set with median from train set
for col in plot_col:
    df_test[col] = df_test[col].fillna(values_dict[col])
    
df_test.isnull().sum()

In [ ]:
# Drop columns
df_test = df_test.drop(['Insulin', 'SkinThickness'], axis=1)
df_test.head(2)

In [ ]:
# Check for missings AGAIN
df_test.isnull().sum().sum()

### Define Features and Target

We separate the features from the target. We do not scale here: scaling goes inside the modelling pipeline below, so it is refit on the training part of each cross-validation fold.

In [ ]:
# Split into features and target 
X_train = df_train.drop("Outcome", axis=1)
y_train = df_train.Outcome
print(X_train.shape)

In [ ]:
# Split test set into features and target
X_test = df_test.drop("Outcome", axis=1)
y_test = df_test.Outcome
print(X_test.shape)

### Handling Class Imbalance

As we have a binary outcome it is always good practice to analyze how balanced the output classes are. 

In [ ]:
# Visual representation of target variable
sns.countplot(x=df_train.Outcome).set_title("Representation of target variable");

We see the classes can not be considered as balanced, so it can help to apply a balancing method. Here we use **SMOTE** (Synthetic Minority Over-sampling Technique) from the `imblearn` package. Instead of simply duplicating minority-class rows the way random oversampling does, SMOTE creates new synthetic examples by interpolating between a minority-class sample and its nearest minority-class neighbours.

Important: we only oversample the **training** data, and we do it **inside the pipeline** (see below) so it runs per cross-validation fold. The validation and test sets keep their real, imbalanced distribution. The cell below applies SMOTE once on the full training set purely to illustrate what it does to the class counts.

See [SMOTE in the imbalanced-learn documentation](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html) for the details and parameters.

In [ ]:
# For illustration only: SMOTE balances the classes. In the modelling below it is
# applied inside the pipeline (per CV fold), not here, so there is no data leakage.
X_demo, y_demo = SMOTE(random_state=42).fit_resample(X_train, y_train)

In [ ]:
# Visual representation of the resampled target variable (illustration only)
sns.countplot(x=y_demo).set_title("Class balance after SMOTE (illustration only)");

### Build the Modelling Pipeline

To avoid data leakage we put scaling and SMOTE inside a single pipeline together with the model. During cross-validation and grid search, the scaler and SMOTE are refit on the training part of each fold only, while the validation fold keeps its real (imbalanced) distribution. We use `imblearn`'s `Pipeline` rather than scikit-learn's, because a sampler like SMOTE must act only during `fit` and be skipped at predict time.

In [ ]:
from imblearn.pipeline import Pipeline

def make_pipeline(model):
    # Scaling and SMOTE live inside the pipeline, so they are refit on the training
    # portion of each CV fold only. SMOTE is skipped at predict time, so the
    # validation fold and the test set keep their real class distribution.
    return Pipeline([
        ("scaler", MinMaxScaler()),
        ("smote", SMOTE(random_state=42)),
        ("model", model),
    ])

## Steps 6 to 8 - Model Training and Hyperparameter Tuning

Feature engineering, model training, hyperparameter tuning, and validation form an iterative loop. We start from a simple baseline, then compare several model types and tune them with grid search. Every model is trained and tuned through the leakage-safe pipeline using stratified 5-fold cross-validation (so each fold keeps the real class balance), and we select on the F2 metric chosen in Step 1 (recall weighted higher than precision).

In [ ]:
# Define fbeta score with higher weighted recall
ftwo_scorer = make_scorer(fbeta_score, beta=2)

# Define dictionary with several interesting metrics
scorer_dict = {"ftwo_scorer": make_scorer(fbeta_score, beta=2), "accuracy": "accuracy", "precision": "precision", "recall": "recall"}

# Cross-validation strategy used everywhere below. For a classifier an integer cv
# already gives stratified folds, but we make it explicit and add shuffling with a
# fixed seed for reproducibility. Stratification is applied to the real labels, so
# each fold keeps the true class balance; SMOTE runs inside the pipeline per fold.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Baseline Model

Before we start training our first model, we should define a proper baseline model. This baseline model represents a educated guess and acts as a benchmark for any further models to beat. From the exploration of our data we can see that the glucose value is a good indicator if someone has diabetes or not. We will choose a value of 130 as the cutoff. For our baseline model we will predict that everyone with a glucose value higher than 130 suffers from diabetes. 

In [ ]:
# Glucose value for both outcomes
sns.boxplot(x=df_train.Outcome, y=df_train.Glucose);

In [ ]:
# Defining baseline model
def baseline_model(df):
    y_pred = [1 if x > 130 else 0 for x in df.Glucose]
    return y_pred

In [ ]:
# Make predictions 
y_pred = baseline_model(X_test)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, cmap="YlGnBu_r", annot=True, fmt=".0f");

In [ ]:
print("Fbeta score with higher weighted recall: ", round(fbeta_score(df_test.Outcome, y_pred, beta=2), 3))

In [ ]:
print(classification_report(df_test.Outcome, y_pred))

Our baseline model reaches a accuracy of 71% and what is even more important a **recall of 58%**. That is the value we have to beat with our more sophisticated models.

### Predictive Modeling

After we got our benchmark via the baseline model, we can now test different model types to figure out which ones can handle our data. We then tune all three with grid search and compare them. 

In [ ]:
def model_evaluation(clf, scoring, X_train, y_train):
    # return_train_score=True lets us compare train vs validation scores, so a large
    # train-validation gap (a sign of overfitting) is already visible at this stage
    scores = cross_validate(clf, X_train, y_train, scoring=scoring, cv=cv, n_jobs=-1, return_train_score=True)
    results = {key: [value.mean().round(4), value.std().round(4)] for key, value in scores.items()}

    del results['fit_time']
    del results['score_time']

    return results

In [ ]:
# Compare model types, each wrapped in the leakage-safe pipeline.
# For F2 we print both the validation and the train score: a big train-validation
# gap is an early warning of overfitting, before we even reach the final test set.
list_of_clf = [LogisticRegression(max_iter=1000), KNeighborsClassifier(), DecisionTreeClassifier(max_depth=5, random_state=42)]

for clf in list_of_clf:
    results = model_evaluation(make_pipeline(clf), scorer_dict, X_train, y_train)
    print(clf)
    print("Accuracy  validation (mean, std):", results["test_accuracy"])
    print("Recall    validation (mean, std):", results["test_recall"])
    print("Precision validation (mean, std):", results["test_precision"])
    print("F2        validation (mean, std):", results["test_ftwo_scorer"])
    print("F2        train      (mean, std):", results["train_ftwo_scorer"])
    print("----"*10)

On the F2 metric, KNN and logistic regression come out close together on the validation folds (within one standard deviation of each other), while the decision tree trails. The train-versus-validation gap already flags overfitting: the decision tree and KNN both score noticeably higher on the training folds than on validation, whereas logistic regression's train and validation scores are close. (We cap the comparison tree at `max_depth=5`; left unconstrained it fits the training folds perfectly.) We tune all three below: logistic regression and KNN as the strongest models, and the decision tree to show hyperparameter tuning and how to rein in overfitting, which is what motivates trying an ensemble such as a Random Forest later.

#### Hyperparameter Tuning of Decision Tree

In [ ]:
# Define param grid (keys prefixed with the pipeline step name 'model')
param_grid = {'model__criterion':  ['gini', 'entropy'],
              'model__max_depth': np.arange(5,70,5),
              'model__min_samples_split': np.arange(5,30, 5)}

# Tune the whole pipeline so scaling and SMOTE are refit inside every CV fold
dec_tree_gs = GridSearchCV(make_pipeline(DecisionTreeClassifier(random_state=42)), param_grid, cv=cv, verbose=1, n_jobs=-1, scoring=ftwo_scorer)

dec_tree_gs.fit(X_train, y_train)

In [ ]:
# Returning best F2 score after GridSearch and the best parameter combination
print("                Best score:", dec_tree_gs.best_score_.round(4))
print("Best parameter combination:", dec_tree_gs.best_params_)
dec_tree_best = dec_tree_gs.best_estimator_

#### Hyperparameter Tuning of KNN

In [ ]:
# In case of knn the parameters to be tuned are n_neighbors and the distance metric p
param_grid = {'model__n_neighbors': np.arange(2,50),
             'model__p': [1, 2]}

# Tune the whole pipeline so scaling and SMOTE are refit inside every CV fold
knn_gs = GridSearchCV(make_pipeline(KNeighborsClassifier()), param_grid, cv=cv, verbose=1, n_jobs=-1, scoring=ftwo_scorer)

knn_gs.fit(X_train, y_train)

In [ ]:
# Returning best F2 score after GridSearch and the best parameter combination
print("                Best score:", knn_gs.best_score_.round(4))
print("Best parameter combination:", knn_gs.best_params_)
knn_best = knn_gs.best_estimator_

#### Hyperparameter Tuning of Logistic Regression

In [ ]:
# For logistic regression, the parameters to be tuned are 
# the regularization strength C and the l1_ratio 
# (the mix between l1 and l2 regularization, 
# with l1_ratio=0 meaning pure l2 and l1_ratio=1 meaning pure l1).
param_grid = {'model__C': np.logspace(-3, 2, 6),
              'model__l1_ratio': [0, 0.5, 1]}

logreg_gs = GridSearchCV(make_pipeline(LogisticRegression(solver='saga', max_iter=5000)), param_grid, cv=cv, verbose=1, n_jobs=-1, scoring=ftwo_scorer)

logreg_gs.fit(X_train, y_train)

In [ ]:
# Returning best F2 score after GridSearch and the best parameter combination
print("                Best score:", logreg_gs.best_score_.round(4))
print("Best parameter combination:", logreg_gs.best_params_)
logreg_best = logreg_gs.best_estimator_

## Step 9 - Calculate Test Score

We evaluate the tuned models on the held-out test set using the F2 metric chosen in Step 1, alongside the full classification report and confusion matrix so we can see precision, recall, and the actual error counts.

### Decision Tree

As the train-versus-validation gap in the comparison already hinted, decision trees tend to overfit, and it is confirmed here: much stronger scores on the train data than on the test data. One option would be to further regularize the model, or we can try to combat overfitting by using an ensemble model which is based on decision trees (e.g. Random Forest).

In [ ]:
# Confusion matrix and classification report for the best decision tree pipeline
y_train_pred_dt = dec_tree_best.predict(X_train)

print("Decision Tree on train data")
print("==="*20)
print("fbeta score:", round(fbeta_score(y_train, y_train_pred_dt, beta=2), 4))
print("---"*20)
print(classification_report(y_train, y_train_pred_dt))
print("---"*20)

cm = confusion_matrix(y_train, y_train_pred_dt)
sns.heatmap(cm, cmap="YlGnBu_r", annot=True, fmt=".0f");

In [ ]:
# Confusion matrix and classification report for the best decision tree pipeline
y_test_pred_dt = dec_tree_best.predict(X_test)

print("Decision Tree on test data")
print("==="*20)
print("fbeta score:", round(fbeta_score(y_test, y_test_pred_dt, beta=2), 4))
print("---"*20)
print(classification_report(y_test, y_test_pred_dt))
print("---"*20)

cm = confusion_matrix(y_test, y_test_pred_dt)
sns.heatmap(cm, cmap="YlGnBu_r", annot=True, fmt=".0f");

### KNN

Our KNN model is also overfitting. That is one of the issues one might encounter when balancing the data with oversampling. But it is still performing better on the test set than the decision tree and it is also beating the benchmark from our baseline model!

In [ ]:
# Confusion matrix and classification report for the best knn pipeline
y_train_pred_knn = knn_best.predict(X_train)

print("KNN on train data")
print("==="*20)
print("fbeta score:", round(fbeta_score(y_train, y_train_pred_knn, beta=2), 4))
print("---"*20)
print(classification_report(y_train, y_train_pred_knn))
print("---"*20)

cm = confusion_matrix(y_train, y_train_pred_knn)
sns.heatmap(cm, cmap="YlGnBu_r", annot=True, fmt=".0f");

In [ ]:
# Confusion matrix and classification report for the best knn pipeline
y_test_pred_knn = knn_best.predict(X_test)

print("KNN on test data")
print("==="*20)
print("fbeta score:", round(fbeta_score(y_test, y_test_pred_knn, beta=2), 4))
print("---"*20)
print(classification_report(y_test, y_test_pred_knn))
print("---"*20)

cm = confusion_matrix(y_test, y_test_pred_knn)
sns.heatmap(cm, cmap="YlGnBu_r", annot=True, fmt=".0f");

### Logistic Regression

Tuned with grid search (best: `C = 1.0`, `l1_ratio = 0.5`, a balanced elasticnet penalty), logistic regression reaches an F2 of about 0.69 on the test set, essentially tied with KNN, but with a more even error profile (recall 0.71, precision 0.64, accuracy 0.76). It is also the only one of the three that barely overfits: its train and test F2 are almost identical (0.70 vs 0.69), matching the small train-versus-validation gap we saw in the comparison. That stability, together with its simplicity and interpretability, makes it a strong choice here, even though KNN edges it slightly on F2.

In [ ]:
# Confusion matrix and classification report for the best logistic regression pipeline
y_train_pred_lr = logreg_best.predict(X_train)

print("Logistic Regression on train data")
print("==="*20)
print("fbeta score:", round(fbeta_score(y_train, y_train_pred_lr, beta=2), 4))
print("---"*20)
print(classification_report(y_train, y_train_pred_lr))
print("---"*20)

cm = confusion_matrix(y_train, y_train_pred_lr)
sns.heatmap(cm, cmap="YlGnBu_r", annot=True, fmt=".0f");

In [ ]:
# Confusion matrix and classification report for the best logistic regression pipeline
y_test_pred_lr = logreg_best.predict(X_test)

print("Logistic Regression on test data")
print("==="*20)
print("fbeta score:", round(fbeta_score(y_test, y_test_pred_lr, beta=2), 4))
print("---"*20)
print(classification_report(y_test, y_test_pred_lr))
print("---"*20)

cm = confusion_matrix(y_test, y_test_pred_lr)
sns.heatmap(cm, cmap="YlGnBu_r", annot=True, fmt=".0f");

### Error Analysis

Let's see where our logistic regression model has problems differentiating between the classes.

In [ ]:
correct = []
for pred, true in zip(y_test_pred_lr, y_test):
    if pred == true and pred==1:
        correct.append("TP")
    if pred == true and pred==0:
        correct.append("TN")
    if pred != true and pred==1:
        correct.append("FP")
    if pred != true and pred==0:
        correct.append("FN")

df_test["correct"] = correct
df_test.head(2)

In [ ]:
# Create exemplary scatterplot for two features
plt.subplots(figsize=(13,8))
sns.scatterplot(data=df_test, x="Glucose", y="BMI", hue="correct", style="Outcome");

From a plot like the one above we can try to draw some conclusions:
- people with too high glucose concentration in blood will generally be predicted as diabetes patients (even if they are not (FP))
- people with relatively low BMIs and Glucose levels will usually be predicted as healthy

Of course the outcome depends on more than those features. We could also have a closer look at other feature combinations to see if we can find patterns.

## Step 10 - Deploy and Monitor

Deployment and monitoring are beyond the scope of this challenge. In practice you would put the chosen model into production behind the same preprocessing steps, then watch its recall on new patients over time and retrain if performance drifts.